# 01 · Preprocessing — Corn Yield Panel

Merge the raw CSVs in `data/raw/` into a single **county × year** panel and save it to
`data/processed/corn_panel.parquet`.

**Final key:** `(stco, year)`

**Final columns:** `stco, year, corn` (target) · 121 `gdd*` bins · `ppt` · 10 soil cols · `A_c` (farmland size)

### Processing rules
1. **Soil** files have no `year`. Attach the era-appropriate soil snapshot per year:
   `1981-1995 -> soil1992`, `1996-2003 -> soil2001`, `2004-2008 -> soil2006`, `2009-2015 -> soil2011`.
2. **gridInfo** has many grids per county -> sum `numAg*` by `stco`, then apply the *same*
   era mapping to pick one `numAg` per year as `A_c` (available farmland).
3. Drop rows where `corn` is NA (non-corn counties, e.g. arid West).
4. Only the **MarAug** climate windows are used (AprOct ignored).

> Raw files under `data/raw/` are **never modified**. Output goes only to `data/processed/`.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

RAW = Path('..') / 'data' / 'raw'
PROC = Path('..') / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

# Era mapping used for BOTH soil and gridInfo (data-manual rule).
ERA_BINS = [(1981, 1995, 1992), (1996, 2003, 2001), (2004, 2008, 2006), (2009, 2015, 2011)]

def year_to_era(y):
    for lo, hi, era in ERA_BINS:
        if lo <= y <= hi:
            return era
    return np.nan

SOIL_COLS = ['whc', 'sand', 'silt', 'clay', 'om', 'kwfactor', 'kffactor', 'spH', 'slope', 'tfactor']
print('pandas', pd.__version__, '| raw dir exists:', RAW.exists())

pandas 2.3.3 | raw dir exists: False


## Step 1 — Base panel from `yielddata.csv`

`yielddata` is a complete, duplicate-free `(stco, year)` grid (3070 counties x 35 years), so we
use it as the merge base. We keep only `corn` as the target and attach the `era` helper column.

In [6]:
yld = pd.read_csv(RAW / 'yielddata.csv')
print('yielddata raw shape:', yld.shape)
print('duplicate (stco,year):', yld.duplicated(['stco', 'year']).sum())

base = yld[['stco', 'year', 'corn']].copy()
base['era'] = base['year'].apply(year_to_era)

print('base shape:', base.shape, '| era NA:', base['era'].isna().sum())
print('corn NA in base:', base['corn'].isna().sum())
base.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/yielddata.csv'

## Step 2 — Temperature exposure `gddMarAug.csv` (121 bins)

The source has 130 junk rows with `year = NaN` (7 counties, all non-corn) — we drop them so
`(stco, year)` is a clean unique key, then left-merge onto the base.

In [ ]:
gdd = pd.read_csv(RAW / 'gddMarAug.csv')
print('gdd raw shape:', gdd.shape, '| year NA rows:', gdd['year'].isna().sum())

gdd = gdd.dropna(subset=['year']).copy()
gdd['year'] = gdd['year'].astype(int)
GDD_COLS = [c for c in gdd.columns if c not in ('stco', 'year')]
print('gdd cleaned shape:', gdd.shape, '| n gdd bins:', len(GDD_COLS),
      '| dup key:', gdd.duplicated(['stco', 'year']).sum())

panel = base.merge(gdd, on=['stco', 'year'], how='left')
print('after gdd merge:', panel.shape, '| gdd0 NA:', panel['gdd0'].isna().sum())
panel.head()

gdd raw shape: (107450, 123) | year NA rows: 130
gdd cleaned shape: (107320, 123) | n gdd bins: 121 | dup key: 0
after gdd merge: (107450, 125) | gdd0 NA: 130


,stco,year,corn,era,gddm60,gddm59,gddm58,gddm57,gddm56,gddm55,...,gddp51,gddp52,gddp53,gddp54,gddp55,gddp56,gddp57,gddp58,gddp59,gddp60
0,1001,1981,17.3,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1003,1981,101.4,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1005,1981,37.9,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1007,1981,NaN,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1009,1981,53.8,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Step 3 — Precipitation `pptMarAug.csv`

Same `year = NaN` junk rows as gdd — drop them, then left-merge `ppt`.

In [ ]:
ppt = pd.read_csv(RAW / 'pptMarAug.csv')
print('ppt raw shape:', ppt.shape, '| year NA rows:', ppt['year'].isna().sum())

ppt = ppt.dropna(subset=['year']).copy()
ppt['year'] = ppt['year'].astype(int)
print('ppt cleaned shape:', ppt.shape, '| dup key:', ppt.duplicated(['stco', 'year']).sum())

panel = panel.merge(ppt, on=['stco', 'year'], how='left')
print('after ppt merge:', panel.shape, '| ppt NA:', panel['ppt'].isna().sum())
panel.head()

ppt raw shape: (107450, 3) | year NA rows: 130
ppt cleaned shape: (107320, 3) | dup key: 0
after ppt merge: (107450, 126) | ppt NA: 130


,stco,year,corn,era,gddm60,gddm59,gddm58,gddm57,gddm56,gddm55,...,gddp52,gddp53,gddp54,gddp55,gddp56,gddp57,gddp58,gddp59,gddp60,ppt
0,1001,1981,17.3,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,620.865553
1,1003,1981,101.4,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,679.832892
2,1005,1981,37.9,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,567.068221
3,1007,1981,NaN,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,561.390156
4,1009,1981,53.8,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,652.470394


## Step 4 — Soil (era-mapped)

Stack the four soil snapshots, each tagged with its `era`, then merge on `(stco, era)` so every
`(stco, year)` row gets the soil snapshot appropriate for that year.

In [ ]:
soil_all = pd.concat(
    [pd.read_csv(RAW / f'soil{era}.csv').assign(era=era) for era in (1992, 2001, 2006, 2011)],
    ignore_index=True,
)
print('soil_all shape:', soil_all.shape, '| dup (stco,era):', soil_all.duplicated(['stco', 'era']).sum())
print('soil NA per source col:')
print(soil_all[SOIL_COLS].isna().sum())

panel = panel.merge(soil_all, on=['stco', 'era'], how='left')
print()
print('after soil merge:', panel.shape, '| whc NA:', panel['whc'].isna().sum())
panel.head()

soil_all shape: (12280, 12) | dup (stco,era): 0
soil NA per source col:
whc          23
sand         23
silt         23
clay         23
om           23
kwfactor     23
kffactor     23
spH          23
slope       601
tfactor      23
dtype: int64

after soil merge: (107450, 136) | whc NA: 178


,stco,year,corn,era,gddm60,gddm59,gddm58,gddm57,gddm56,gddm55,...,whc,sand,silt,clay,om,kwfactor,kffactor,spH,slope,tfactor
0,1001,1981,17.3,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,24.783429,59.681402,20.048434,20.270163,111.806781,0.223065,0.226399,5.139626,51.036285,4.950716
1,1003,1981,101.4,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,23.538841,59.091992,21.805165,19.102843,185.667551,0.250390,0.256134,4.940044,106.221047,4.936530
2,1005,1981,37.9,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,21.483670,61.744352,16.410103,21.845545,88.429680,0.212322,0.212400,5.127343,65.980025,4.926214
3,1007,1981,NaN,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,25.890773,50.707970,24.242096,25.049933,71.572749,0.244816,0.256716,4.957079,41.041487,4.861003
4,1009,1981,53.8,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,18.282575,37.500558,33.404699,29.094743,79.915417,0.245428,0.296248,4.914515,32.146565,3.174986


## Step 5 — Available farmland `A_c` from `gridInfo.csv`

gridInfo has many grid rows per county. Sum `numAg*` by `stco` to get county-level farmland,
then melt to long form and apply the era mapping so each year picks the right `numAg` as `A_c`.

In [ ]:
grid = pd.read_csv(RAW / 'gridInfo.csv')
print('gridInfo raw shape:', grid.shape, '| unique stco:', grid['stco'].nunique())

num_cols = ['numAg1992', 'numAg2001', 'numAg2006', 'numAg2011']
grid_agg = grid.groupby('stco', as_index=False)[num_cols].sum()
print('grid_agg (county-level) shape:', grid_agg.shape)

ac = grid_agg.melt(id_vars='stco', value_vars=num_cols, var_name='numcol', value_name='A_c')
ac['era'] = ac['numcol'].str.replace('numAg', '').astype(int)
ac = ac.drop(columns='numcol')
print('A_c long shape:', ac.shape, '| dup (stco,era):', ac.duplicated(['stco', 'era']).sum())

panel = panel.merge(ac, on=['stco', 'era'], how='left')
print('after A_c merge:', panel.shape, '| A_c NA:', panel['A_c'].isna().sum())
panel.head()

gridInfo raw shape: (455582, 6) | unique stco: 3070
grid_agg (county-level) shape: (3070, 5)
A_c long shape: (12280, 3) | dup (stco,era): 0
after A_c merge: (107450, 137) | A_c NA: 0


,stco,year,corn,era,gddm60,gddm59,gddm58,gddm57,gddm56,gddm55,...,sand,silt,clay,om,kwfactor,kffactor,spH,slope,tfactor,A_c
0,1001,1981,17.3,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,59.681402,20.048434,20.270163,111.806781,0.223065,0.226399,5.139626,51.036285,4.950716,352723
1,1003,1981,101.4,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,59.091992,21.805165,19.102843,185.667551,0.250390,0.256134,4.940044,106.221047,4.936530,1111249
2,1005,1981,37.9,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,61.744352,16.410103,21.845545,88.429680,0.212322,0.212400,5.127343,65.980025,4.926214,476667
3,1007,1981,NaN,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,50.707970,24.242096,25.049933,71.572749,0.244816,0.256716,4.957079,41.041487,4.861003,135346
4,1009,1981,53.8,1992,0.0,0.0,0.0,0.0,0.0,0.0,...,37.500558,33.404699,29.094743,79.915417,0.245428,0.296248,4.914515,32.146565,3.174986,497673


## Step 6 — Drop non-corn rows and finalize column order

In [ ]:
rows_before = len(panel)
panel = panel.dropna(subset=['corn']).copy()
rows_after = len(panel)
print(f'corn NA drop: {rows_before} -> {rows_after}  (removed {rows_before - rows_after})')

final_cols = ['stco', 'year', 'corn'] + GDD_COLS + ['ppt'] + SOIL_COLS + ['A_c']
corn_panel = panel[final_cols].reset_index(drop=True)
print('final panel shape:', corn_panel.shape,
      '| expected columns 3+121+1+10+1 =', 3 + 121 + 1 + 10 + 1)
corn_panel.head()

corn NA drop: 107450 -> 70721  (removed 36729)
final panel shape: (70721, 136) | expected columns 3+121+1+10+1 = 136


,stco,year,corn,gddm60,gddm59,gddm58,gddm57,gddm56,gddm55,gddm54,...,sand,silt,clay,om,kwfactor,kffactor,spH,slope,tfactor,A_c
0,1001,1981,17.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,59.681402,20.048434,20.270163,111.806781,0.223065,0.226399,5.139626,51.036285,4.950716,352723
1,1003,1981,101.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,59.091992,21.805165,19.102843,185.667551,0.250390,0.256134,4.940044,106.221047,4.936530,1111249
2,1005,1981,37.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,61.744352,16.410103,21.845545,88.429680,0.212322,0.212400,5.127343,65.980025,4.926214,476667
3,1009,1981,53.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,37.500558,33.404699,29.094743,79.915417,0.245428,0.296248,4.914515,32.146565,3.174986,497673
4,1011,1981,22.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,39.390671,25.923337,34.685992,120.902921,0.230438,0.232822,5.529810,46.056167,4.988563,295876


## Step 7 — Save to `data/processed/corn_panel.parquet`

In [ ]:
out_path = PROC / 'corn_panel.parquet'
corn_panel.to_parquet(out_path, index=False)
print('saved:', out_path.resolve())

# sanity round-trip
check = pd.read_parquet(out_path)
print('round-trip shape:', check.shape)

saved: C:\Users\3425e\corn-project\data\processed\corn_panel.parquet
round-trip shape: (70721, 136)


## Step 8 — Summary

Final row count, county count, year range, corn-NA before/after, and any remaining missingness.

In [ ]:
print('=' * 55)
print('CORN PANEL - SUMMARY')
print('=' * 55)
print(f'final rows            : {len(corn_panel):,}')
print(f'unique counties (stco): {corn_panel["stco"].nunique():,}')
print(f'year range            : {corn_panel["year"].min()} - {corn_panel["year"].max()}')
print(f'columns               : {corn_panel.shape[1]}')
print()
print(f'rows before corn-NA drop : {rows_before:,}')
print(f'rows after  corn-NA drop : {rows_after:,}')
print(f'rows removed (corn NA)   : {rows_before - rows_after:,}')
print()
miss = corn_panel.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
if len(miss):
    print('remaining missing values by column (inherited from source data):')
    print(miss.to_string())
else:
    print('no remaining missing values.')

CORN PANEL - SUMMARY
final rows            : 70,721
unique counties (stco): 2,644
year range            : 1981 - 2015
columns               : 136

rows before corn-NA drop : 107,450
rows after  corn-NA drop : 70,721
rows removed (corn NA)   : 36,729

remaining missing values by column (inherited from source data):
slope       2061
sand           4
whc            4
kwfactor       4
silt           4
om             4
clay           4
spH            4
kffactor       4
tfactor        4
gddm4          1


In [ ]:
panel.groupby('year')['stco'].nunique()

year
1981    2387
1982    2380
1983    2340
1984    2340
1985    2329
1986    2294
1987    2279
1988    2244
1989    2234
1990    2169
1991    2198
1992    2229
1993    2144
1994    2148
1995    2067
1996    2111
1997    2076
1998    2091
1999    2067
2000    2046
2001    1990
2002    2053
2003    2024
2004    1981
2005    1970
2006    1921
2007    1896
2008    1584
2009    1553
2010    1717
2011    1633
2012    1686
2013    1545
2014    1562
2015    1433
Name: stco, dtype: int64